# Example 1: Simple Format - ALS PBMC Dataset

This notebook demonstrates processing a traditional GEO dataset where each sample has a single expression file.

**Dataset**: GSE244263 (ALS vs Control PBMC samples)  
**Format**: One .txt.gz file per sample  
**Samples**: 3 samples (2 ALS, 1 Control)

In [ ]:
import sys
import os
sys.path.append('../../')  # Add repo to path

from anndata_compiler import GEOAnndataCompiler
import pandas as pd

## 1. Examine the Data Structure

In [ ]:
# Look at the files
data_dir = '../data/simple_format_ALS'
print("Files in data directory:")
for f in os.listdir(data_dir):
    print(f"  {f}")

# Look at metadata
metadata = pd.read_csv(f'{data_dir}/metadata.csv')
print(f"\nMetadata ({metadata.shape[0]} samples):")
print(metadata[['Sample_name', 'Sample_geo_accession', 'Disease_state', 'Sample_ID']])

## 2. Configure the Compiler

For simple format, we just need basic configuration:

In [ ]:
config = {
    'raw_data_dir': data_dir,
    'metadata_file': f'{data_dir}/metadata.csv',
    'output_file': './als_compiled_example.h5ad',
    'sample_id_column': 'Sample_ID',
    
    # Processing parameters
    'max_cells_per_sample': 500,  # Small for demo
    'target_sum': 1e4,
    'n_top_genes': 2000,
    'delimiter': 'whitespace',  # .txt files are typically tab/space separated
    'optimize_params': True,
    
    # Metadata filtering - only include disease state
    'metadata_columns': ['Disease_state', 'Source']
}

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 3. Run the Compilation Pipeline

In [ ]:
# Initialize compiler
compiler = GEOAnndataCompiler(config)

# Run full pipeline
adata = compiler.run_full_pipeline(
    plot_colors=['leiden', 'Disease_state', 'sample_id']
)

print(f"\nFinal dataset: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"Samples: {adata.obs['sample_id'].unique()}")
print(f"Disease states: {adata.obs['Disease_state'].unique()}")

## 4. Explore the Results

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Basic info
print("AnnData structure:")
print(adata)
print(f"\nObservation columns: {list(adata.obs.columns)}")
print(f"Variable columns: {list(adata.var.columns)}")

# Sample composition
sample_counts = adata.obs.groupby(['sample_id', 'Disease_state']).size().unstack(fill_value=0)
print(f"\nCells per sample:")
print(sample_counts)

In [ ]:
# UMAP colored by different factors
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, frameon=False)
axes[0].set_title('Leiden Clusters')

sc.pl.umap(adata, color='Disease_state', ax=axes[1], show=False, frameon=False)
axes[1].set_title('Disease State')

sc.pl.umap(adata, color='sample_id', ax=axes[2], show=False, frameon=False)
axes[2].set_title('Sample ID')

plt.tight_layout()
plt.show()

## Summary

This example showed:
- Auto-detection of simple format
- Processing traditional GEO files (.txt.gz)
- Selective metadata inclusion
- Standard scanpy preprocessing pipeline
- Quality control and visualization

The compiled AnnData object is ready for downstream analysis!